# Продукт и рекомендации

Ноутбук готовит данные для дашборда состояния продукта и воронки
рекомендаций. Одна строка продуктовой витрины соответствует одному
условному дню.

## Определения

- активный пользователь - пользователь с любым событием в этот день;
- слушатель - пользователь хотя бы с одним прослушиванием;
- Listen+ - прослушано больше 50% трека;
- повтор - прослушано больше 100% трека;
- рекомендация - прослушивание с `is_organic = 0`.

В данных нет показов и кликов по рекомендательной ленте, поэтому
воронка начинается с фактического прослушивания.

In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STAGE_DB = PROJECT_ROOT / "data" / "interim" / "yambda_stage.duckdb"
MARTS = PROJECT_ROOT / "data" / "processed"
assert STAGE_DB.exists(), "Сначала выполните ноутбук 01_source_quality.ipynb"
MARTS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
con = duckdb.connect()

## Сборка витрин

In [2]:
from src.product_mart import build_product_mart, build_recommendation_funnel

PRODUCT_MART = MARTS / "mart_product_day.parquet"
FUNNEL_MART = MARTS / "mart_recommendation_funnel.parquet"

product_result = build_product_mart(STAGE_DB, PRODUCT_MART)
funnel_result = build_recommendation_funnel(STAGE_DB, FUNNEL_MART)
{"продукт": product_result, "воронка": funnel_result}

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'продукт': {'rows': 301, 'events': 47790449, 'listens': 46467212},
 'воронка': {'rows': 4}}

## Ключевые показатели продукта

In [3]:
product_summary = con.execute(f'''
SELECT
    count(*) AS days,
    sum(events) AS events,
    sum(listens) AS listens,
    median(active_users) AS median_dau,
    sum(sessions) AS sessions,
    sum(play_seconds) / sum(active_users) / 60.0 AS minutes_per_active_user,
    sum(listen_plus) * 100.0 / sum(listens) AS listen_plus_pct,
    sum(recommendation_listens) * 100.0 / sum(listens)
        AS recommendation_pct,
    sum(replays) * 100.0 / sum(listens) AS replay_pct,
    sum(likes) * 1000.0 / sum(listens) AS likes_per_1000,
    sum(dislikes) * 1000.0 / sum(listens) AS dislikes_per_1000
FROM read_parquet('{PRODUCT_MART.as_posix()}')
''').df().round(2)

product_summary.rename(columns={
    "days": "дни", "events": "события", "listens": "прослушивания",
    "median_dau": "медианный DAU", "sessions": "сессии",
    "minutes_per_active_user": "минуты на активного пользователя",
    "listen_plus_pct": "Listen+, %",
    "recommendation_pct": "рекомендации, %", "replay_pct": "повторы, %",
    "likes_per_1000": "лайки на 1000",
    "dislikes_per_1000": "дизлайки на 1000",
})

,дни,события,прослушивания,медианный DAU,сессии,минуты на активного пользователя,"Listen+, %","рекомендации, %","повторы, %",лайки на 1000,дизлайки на 1000
0,301,"47,790,449.00","46,467,212.00","3,476.00","2,529,801.00",95.12,63.21,48.34,0.47,18.97,2.32


## Рекомендации против органики

In [4]:
source_quality = con.execute(f'''
SELECT
    'рекомендации' AS source,
    sum(recommendation_listens) AS listens,
    sum(recommendation_listen_plus) * 100.0
        / sum(recommendation_listens) AS listen_plus_pct,
    sum(recommendation_replays) * 100.0
        / sum(recommendation_listens) AS replay_pct
FROM read_parquet('{PRODUCT_MART.as_posix()}')
UNION ALL
SELECT
    'органика',
    sum(organic_listens),
    sum(organic_listen_plus) * 100.0 / sum(organic_listens),
    sum(organic_replays) * 100.0 / sum(organic_listens)
FROM read_parquet('{PRODUCT_MART.as_posix()}')
''').df().round(2)

source_quality.rename(columns={
    "source": "источник", "listens": "прослушивания",
    "listen_plus_pct": "Listen+, %", "replay_pct": "повторы, %",
})

,источник,прослушивания,"Listen+, %","повторы, %"
0,рекомендации,"22,463,555.00",68.06,0.24
1,органика,"24,003,657.00",58.67,0.68


## Воронка качества рекомендаций

In [5]:
funnel = con.execute(f'''
SELECT
    step_order,
    step,
    events,
    users,
    event_rate_from_start * 100 AS event_pct_from_start,
    user_rate_from_start * 100 AS user_pct_from_start
FROM read_parquet('{FUNNEL_MART.as_posix()}')
ORDER BY step_order
''').df().round(2)

funnel["этап"] = funnel["step"].map({
    "all_listens": "все прослушивания",
    "recommendation_listens": "прослушивания рекомендаций",
    "recommendation_listen_plus": "рекомендации Listen+",
    "recommendation_replays": "повторы рекомендаций",
})
funnel[["этап", "events", "users", "event_pct_from_start", "user_pct_from_start"]].rename(
    columns={"events": "события", "users": "пользователи",
             "event_pct_from_start": "% событий от начала",
             "user_pct_from_start": "% пользователей от начала"}
)

,этап,события,пользователи,% событий от начала,% пользователей от начала
0,все прослушивания,46467212,9238,100.00,100.00
1,прослушивания рекомендаций,22463555,8789,48.34,95.14
2,рекомендации Listen+,15288084,8588,32.90,92.96
3,повторы рекомендаций,54066,4937,0.12,53.44


## Дневной набор для будущих графиков

In [6]:
daily_preview = con.execute(f'''
SELECT
    day_idx,
    active_users,
    listeners,
    listens,
    sessions,
    round(minutes_per_active_user, 2) AS minutes_per_active_user,
    round(listen_plus_rate * 100, 2) AS listen_plus_pct,
    round(recommendation_share * 100, 2) AS recommendation_pct,
    round(likes_per_1000_listens, 2) AS likes_per_1000,
    round(dislikes_per_1000_listens, 2) AS dislikes_per_1000
FROM read_parquet('{PRODUCT_MART.as_posix()}')
ORDER BY day_idx
LIMIT 10
''').df()
daily_preview

,day_idx,active_users,listeners,listens,sessions,minutes_per_active_user,listen_plus_pct,recommendation_pct,likes_per_1000,dislikes_per_1000
0,0,2653,2542,114247,6088,98.82,66.23,51.49,22.48,1.58
1,1,2421,2336,105991,5428,99.02,65.31,46.48,22.28,1.53
2,2,2320,2225,94411,4830,89.04,62.92,47.07,20.86,2.38
3,3,2591,2474,106734,5900,95.54,66.24,52.26,24.35,2.09
4,4,2646,2525,107749,5928,94.65,66.49,53.18,25.36,2.73
5,5,2628,2523,108422,5904,97.06,67.45,53.13,22.46,2.25
6,6,2638,2538,112489,6097,97.79,65.45,53.65,22.25,2.13
7,7,2721,2612,121472,6349,98.14,62.60,51.70,20.25,2.10
8,8,2511,2409,106697,5452,92.95,62.25,48.16,21.66,2.39
9,9,2385,2295,95115,5219,88.96,64.06,49.78,19.43,1.90


## Проверки

In [7]:
product_check = con.execute(f'''
SELECT
    count(*) = count(DISTINCT day_idx) AS unique_key,
    sum(events) = 47790449 AS events_match,
    sum(listens) = 46467212 AS listens_match,
    min(listen_plus_rate) >= 0 AND max(listen_plus_rate) <= 1
        AS listen_plus_valid,
    min(recommendation_share) >= 0 AND max(recommendation_share) <= 1
        AS recommendation_share_valid
FROM read_parquet('{PRODUCT_MART.as_posix()}')
''').df()
funnel_check = con.execute(f'''
WITH ordered AS (
    SELECT *, lag(events) OVER (ORDER BY step_order) AS previous_events
    FROM read_parquet('{FUNNEL_MART.as_posix()}')
)
SELECT count(*) = 4 AS four_steps,
       bool_and(previous_events IS NULL OR events <= previous_events)
           AS monotonic
FROM ordered
''').df()
assert product_check.all(axis=None) and funnel_check.all(axis=None)
print("Проверки пройдены: ключи, суммы, доли и воронка корректны.")

Проверки пройдены: ключи, суммы, доли и воронка корректны.


## Что готово для дашборда

- карточки DAU, слушателей, прослушиваний, сессий и времени;
- динамика Listen+, рекомендаций, повторов и реакций;
- сравнение качества рекомендаций и органики;
- воронка рекомендаций.

Крайние дни перед сравнением периодов нужно проверять на полноту.

In [8]:
con.close()
print("Соединение закрыто.")

Соединение закрыто.
